## Load the required packages

In [1]:
import pandas as pd
from sciPENN.sciPENN_API import sciPENN_API
import anndata
from scipy.sparse import csc_matrix, csr_matrix
import os
from pathlib import Path
import numpy as np

## Set path

In [2]:
working_dir = Path(os.getcwd())
dataset_dir = f"{working_dir.parents[1]}/datasets/CITE-SLN-Gayoso"
base_dir = f"{working_dir.parents[1]}/outputs/batch variation/208_M2-208_M1/sciPENN"
os.makedirs(base_dir, exist_ok=True)

## Load data

In [3]:
train_data = anndata.read(f"{dataset_dir}/SLN208_M2.h5ad")   
test_data = anndata.read(f"{dataset_dir}/SLN208_M1.h5ad") 
train_data, test_data

(AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

## Select common genes

In [4]:
common_genes = np.intersect1d(train_data.var.index, test_data.var.index)
train_data = train_data[:, common_genes]
test_data = test_data[:, common_genes]

## Select common proteins

In [5]:
common_proteins = np.intersect1d(np.array(train_data.uns["protein_names"]), np.array(test_data.uns["protein_names"]))
if isinstance(train_data.obsm["protein_expression"], pd.DataFrame):
    train_protein_expression = train_data.obsm["protein_expression"]
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = train_protein_expression
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)
else:
    train_protein_expression = pd.DataFrame(train_data.obsm["protein_expression"].todense(), index=train_data.obs.index, columns=train_data.uns["protein_names"])
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = csr_matrix(train_protein_expression.values)
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)

if isinstance(test_data.obsm["protein_expression"], pd.DataFrame):
    test_protein_expression = test_data.obsm["protein_expression"]
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = test_protein_expression
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)
else:
    test_protein_expression = pd.DataFrame(test_data.obsm["protein_expression"].todense(), index=test_data.obs.index, columns=test_data.uns["protein_names"])
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = csr_matrix(test_protein_expression.values)
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)

In [6]:
adata_gene_train = anndata.AnnData(X=csc_matrix(train_data.X), obs=train_data.obs, var=train_data.var)
adata_protein_train = anndata.AnnData(X=train_protein_expression)
adata_protein_train.X = csc_matrix(adata_protein_train.X)

adata_gene_test = anndata.AnnData(X=csc_matrix(test_data.X), obs=test_data.obs, var=test_data.var)
adata_gene_train, adata_protein_train, adata_gene_test

(AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode',
 AnnData object with n_obs × n_vars = 7105 × 198,
 AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode')

In [7]:
sciPENN = sciPENN_API(gene_trainsets=[adata_gene_train], protein_trainsets=[adata_protein_train], gene_test=adata_gene_test, train_batchkeys=None, 
                      test_batchkey=None, type_key=None, min_genes=0)

Searching for GPU
GPU detected, using GPU

QC Filtering Training Genes
QC Filtering Testing Genes

Normalizing Training Cells
Normalizing Testing Cells

Log-Normalizing Training Data
Log-Normalizing Testing Data

Finding HVGs


/home/hongyj/.conda/envs/scipennenv/lib/python3.7/site-packages/anndata/_core/anndata.py:1785: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  [AnnData(sparse.csr_matrix(a.shape), obs=a.obs) for a in all_adatas],



Normalizing Gene Training Data by Batch


100%|██████████| 1/1 [00:00<00:00,  5.43it/s]



Normalizing Protein Training Data by Batch


100%|██████████| 1/1 [00:00<00:00, 18.52it/s]



Normalizing Gene Testing Data by Batch


100%|██████████| 1/1 [00:00<00:00,  5.20it/s]


## Start training

In [8]:
sciPENN.train(quantiles=[0.1, 0.25, 0.75, 0.9], n_epochs=10000, ES_max=12, decay_max=6, decay_step=0.1, lr=10**(-3), weights_dir=base_dir, load=True)

Epoch 0 prediction loss = 1.404
Epoch 1 prediction loss = 1.054
Epoch 2 prediction loss = 1.032
Epoch 3 prediction loss = 1.028
Epoch 4 prediction loss = 1.018
Epoch 5 prediction loss = 1.014
Epoch 6 prediction loss = 1.011
Epoch 7 prediction loss = 1.011
Epoch 8 prediction loss = 1.010
Epoch 9 prediction loss = 1.008
Epoch 10 prediction loss = 1.006
Epoch 11 prediction loss = 1.005
Epoch 12 prediction loss = 1.010
Epoch 13 prediction loss = 1.011
Epoch 14 prediction loss = 1.005
Epoch 15 prediction loss = 1.005
Epoch 16 prediction loss = 1.008
Decaying loss to 0.0001
Epoch 17 prediction loss = 1.000
Epoch 18 prediction loss = 0.997
Epoch 19 prediction loss = 0.995
Epoch 20 prediction loss = 0.997
Epoch 21 prediction loss = 0.999
Epoch 22 prediction loss = 0.998
Epoch 23 prediction loss = 0.997
Decaying loss to 1e-05
Epoch 24 prediction loss = 0.996
Epoch 25 prediction loss = 0.998
Epoch 26 prediction loss = 0.998
Epoch 27 prediction loss = 0.997
Epoch 28 prediction loss = 0.998
Epoch 

## Perform prediction on the query

In [9]:
predicted_test = sciPENN.predict()
predicted_protein_expression = pd.DataFrame(predicted_test.X, index=predicted_test.obs.index, columns=predicted_test.var.index)
predicted_protein_expression.to_csv(f"{base_dir}/test_predicted_protein_expression (sciPENN).csv")
predicted_protein_expression

/home/hongyj/.conda/envs/scipennenv/lib/python3.7/site-packages/sciPENN/Network/Model.py:211: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  imputed_test = AnnData(zeros(shape = (len(cells), len(proteins.var))))


,ADT_4-1BBLigand(CD137L)_A0890,ADT_CD102_A0104,ADT_CD103_A0201,ADT_CD105_A0812,ADT_CD106_A0226,ADT_CD107a(LAMP-1)_A0905,ADT_CD115(CSF-1R)_A0105,ADT_CD117(c-Kit)_A0012,ADT_CD11a_A0595,ADT_CD11b_A0014,...,ADT_TCRVr3_A0210,ADT_TCRbchain_A0120,ADT_TCRr-d_A0121,ADT_TER-119-ErythroidCells_A0122,ADT_TIGIT(Vstm3)_A0848,ADT_TLR4(CD284)-MD2Complex_A0875,ADT_Tim-4_A0567,ADT_XCR1_A0568,ADT_anti-P2RY12_A0415,ADT_integrinb7_A0214
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGCGATCGA-1,0.188444,-0.567495,2.088368,0.232421,-0.089684,-0.074510,-0.030434,0.505460,0.567644,-0.253308,...,0.169681,0.896572,0.232635,0.088154,0.150464,0.183326,-0.167964,-0.034733,0.174819,1.920757
AAACCCACACCTCTAC-1,-0.423384,-0.190193,-0.581127,-0.440532,-0.326477,-0.298228,-0.361092,-0.576740,0.203267,-0.147351,...,-0.637971,-0.936021,-0.370120,-0.266106,-0.678810,-0.259483,-0.270455,-0.121162,-0.334040,-1.120907
AAACCCACACGTCGTG-1,-0.003441,1.454011,-0.322339,0.074658,-0.186915,0.020656,-0.311041,-0.423998,-0.803055,-0.107882,...,-0.222127,-0.673446,0.063900,-0.005280,-0.107840,-0.048253,-0.152286,-0.182600,-0.009380,-0.049320
AAACCCACATGTCTAG-1,-0.324205,1.038129,-0.546896,-0.206643,-0.343147,-0.178257,-0.218454,-0.513134,-0.631775,-0.194414,...,-0.348545,-0.711267,-0.177114,-0.030814,-0.329007,-0.134264,-0.261795,-0.354255,0.007489,-0.187682
AAACCCAGTCAAAGTA-1,-0.253225,0.022266,-0.631884,-0.359364,-0.273105,-0.064808,-0.126166,-0.434414,-0.464832,-0.231762,...,-0.043408,-0.848772,-0.158759,-0.072639,-0.275851,-0.279376,-0.385208,-0.048246,-0.114664,-0.095205
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,-0.238180,0.466300,-0.398439,-0.127520,0.027086,-0.231895,0.021040,-0.160284,-0.742571,-0.272676,...,-0.112406,-0.689543,0.033746,0.115266,-0.179241,-0.066862,-0.130328,-0.305330,-0.113178,0.020112
TTTGTTGGTGCTCTTC-1,0.189118,0.228618,-0.153718,-0.246224,-0.138171,0.176170,-0.035834,0.036781,-0.684287,-0.128659,...,0.308518,-0.665583,0.123348,0.041192,0.263003,0.150117,-0.029356,0.411345,0.202460,0.279879
TTTGTTGTCAGCCTTC-1,-0.034573,-0.561631,-0.164730,-0.044611,-0.199979,-0.068143,-0.112131,0.167566,0.500228,-0.328822,...,0.148118,1.259439,-0.013154,-0.035220,0.144028,-0.001947,-0.256934,0.154750,0.046542,-0.222294
